# EXP-3 — Query Rewriting

User queries are often vague or informal. We ask the LLM to rewrite the query into a cleaner, more specific form before hitting the vector store. Retrieval happens on the rewritten query, not the original.

## Setup

In [11]:
import os
import sys
import time
import mlflow
import pandas as pd
from dotenv import load_dotenv
from datasets import Dataset

load_dotenv()

from langchain_community.document_loaders import DirectoryLoader,TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
import warnings
warnings.filterwarnings("ignore")

print("all imports loaded")


all imports loaded


In [4]:
sys.path.append(os.path.abspath(".."))
from data.policies.qa_dataset import dataset

In [5]:
# langsmith traces every chain.invoke() automatically once these env vars are set
# you will see each run in your langsmith project dashboard in real time
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "HR-RAG-Experiments"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "")

print("langsmith tracing is active")


langsmith tracing is active


In [6]:

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("HR_RAG_Experiments")

print(f"mlflow tracking uri: {MLFLOW_TRACKING_URI}")


mlflow tracking uri: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow


## Load and Prepare Data

In [12]:
# adjust the path and glob pattern to match your folder structure
loader = DirectoryLoader("../data/policies", glob="**/*.md", loader_cls=TextLoader)
docs = loader.load()

print(f"loaded {len(docs)} documents")


loaded 5 documents


In [13]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(docs)
print(f"total chunks: {len(chunks)}")


total chunks: 98


In [14]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
db = FAISS.from_documents(chunks, embeddings)

print("faiss vector store ready")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2303.52it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


faiss vector store ready


In [15]:
print(f"eval set: {len(dataset['question'])} question-answer pairs")

eval set: 10 question-answer pairs


In [23]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant"
)

In [24]:
prompt = ChatPromptTemplate.from_template("""
You are an HR assistant. Use only the context below to answer the question.
If the answer is not present in the context, say you do not know.

Context:
{context}

Question: {question}

Answer:""")


## Helper Functions

In [25]:
def format_docs(docs):
    # joins retrieved chunks into a single string that goes into the prompt context
    return "\n\n".join(doc.page_content for doc in docs)

In [36]:
from ragas.llms import LangchainLLMWrapper
from ragas.run_config import RunConfig

ragas_llm = LangchainLLMWrapper(llm, run_config=RunConfig(max_workers=1))

In [37]:
import nest_asyncio
nest_asyncio.apply()

def evaluate_rag(chain, get_docs_fn, dataset):
    answers = []
    contexts = []

    for question in dataset["question"]:
        try:
            answer = chain.invoke(question)
            docs = get_docs_fn(question)
            answers.append(answer)
            contexts.append([d.page_content for d in docs])
        except Exception as e:
            print(f"  error on: {question[:60]} => {e}")
            answers.append("error")
            contexts.append(["error"])

    eval_dataset = Dataset.from_dict({
        "question": dataset["question"],
        "answer": answers,
        "contexts": contexts,
        "ground_truth": dataset["answer"],
    })

    # max_workers=1 means sequential — no parallel calls, no rate limit hits
    # timeout=120 because Groq free tier can be slow sometimes
    run_cfg = RunConfig(max_workers=1, timeout=120)

    result = evaluate(
        eval_dataset,
        metrics=[faithfulness, context_precision, context_recall],
        llm=ragas_llm,
        embeddings=embeddings,
        run_config=run_cfg,
        raise_exceptions=False,   # if one question fails,we still want to get metrics on the rest of the eval set
    )
    return result

In [38]:
def measure_latency(chain, test_q="What is the leave policy?"):
    start = time.time()
    chain.invoke(test_q)
    return round(time.time() - start, 3)


In [44]:
def log_to_mlflow(run_name, result, latency, retriever_type, top_k=3, extra_params=None):
    with mlflow.start_run(run_name=run_name):
        if hasattr(result, "to_pandas"):
           
            scores = result.to_pandas().mean(numeric_only=True).to_dict()
        else:
            scores = dict(result)

        for k, v in scores.items():
            try:
                mlflow.log_metric(k, float(v))
            except Exception:
                pass

        mlflow.log_metric("latency_seconds", latency)
        mlflow.log_param("retriever_type", retriever_type)
        mlflow.log_param("top_k", top_k)

        if extra_params:
            for k, v in extra_params.items():
                mlflow.log_param(k, str(v))

## Experiment

In [40]:
rewrite_prompt = ChatPromptTemplate.from_template(
    "Rewrite this search query to be more specific for searching an HR policy document.\n\n"
    "Original query: {question}\n\n"
    "Rewritten query:"
)

rewriter = rewrite_prompt | llm | StrOutputParser()
base_retriever = db.as_retriever(search_kwargs={"k": 3})

def rewrite_and_retrieve(question):
    rewritten = rewriter.invoke({"question": question})
    return base_retriever.invoke(rewritten)

chain = (
    {
        "context": RunnableLambda(rewrite_and_retrieve) | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

# testing with a deliberately vague query
print(chain.invoke("tell me about leave"))


Based on the context, here is the information about leave:

- The Leave Management Policy outlines the types of leaves available to all employees of TechCorp India Pvt. Ltd.
- The policy applies to all employees across all departments, levels, and locations.
- Types of leaves are not explicitly mentioned, so it is not specified.
- Policy violations include:
  - Taking leave without approval: Salary deduction + disciplinary action
  - Misrepresentation of reasons: Termination of employment
  - Frequent uninformed absences: Performance Improvement Plan (PIP)
- The contact information for the HR Leave Management Team is:
  - Email: leave.hr@techcorp.in
  - Helpline: 1800-HR-LEAVE (Mon-Fri, 9 AM - 6 PM)
  - HRMS Portal: hrms.techcorp.in
- There is no information about other types of leaves in the provided context.


In [41]:
result = evaluate_rag(chain, rewrite_and_retrieve, dataset)
latency = measure_latency(chain)

Evaluating: 100%|██████████| 30/30 [11:43<00:00, 23.44s/it]


In [43]:
result,latency

({'faithfulness': 0.5000, 'context_precision': 0.5667, 'context_recall': 0.3524},
 3.701)

In [45]:
log_to_mlflow(
    run_name="EXP-3-QUERY-REWRITE",
    result=result,
    latency=latency,
    retriever_type="query-rewrite",
    top_k=3,
)


🏃 View run EXP-3-QUERY-REWRITE at: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow/#/experiments/1/runs/448eb8d5ef2e4970b03581f25bb79115
🧪 View experiment at: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow/#/experiments/1
